In [ ]:
#@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@
#A very short code to explore how we could use sampling to update distributions
#Jeremy Kedziora
#25 February
#@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@


In [ ]:
#@@@@@@@@@@@@@@@@@@@@@@@
#import useful libraries
#@@@@@@@@@@@@@@@@@@@@@@@
from sklearn.neighbors import KernelDensity
import numpy as np
from IPython.display import clear_output
import time
import matplotlib.pyplot as plt


In [ ]:
#@@@@@@@@@@@@@@@@@@@@@@
#make the original data
#@@@@@@@@@@@@@@@@@@@@@@

#@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@
def d_G_init(low = -10.0,hi = 10.0):
    '''A function to generate an initial G dataset -- assumed uniform
        Takes:
            low -- float, the initial lower bound on G
            hi -- float, the initial upper bound on G
        Returns:
            a fitted KernelDensity object representing d_G_s_a
    '''
    data = np.random.uniform(low = low, high = hi, size = 1000)[:, np.newaxis]    #sample uniformly at random
    d_G_s_a = KernelDensity(bandwidth=0.2, kernel='gaussian')    #instantiate the KDE
    d_G_s_a.fit(data)   #fit it
    return d_G_s_a

#@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@
def d_G_update(d_G_s_a,data_new,n_upsample,n_samples,entropy):
    '''A function to update estimates of d_G_s_a
        Takes:
            d_G_s_a -- fitted KDE representing the old distribution
            data_new -- an nx1 array of new G estimates from rollouts
            n_upsample -- int, the number of times to upsample new Gs for the KDE estimate
            n_samples -- int, the number of old Gs to sample
            entropy -- float, fraction of uniform samples to add to maintain entropy
        Returns:
            a fitted KernelDensity object representing d_G_s_a
    '''
    data_old = d_G_s_a.sample(n_samples)    #sample from the OG distribution
    # data_new = np.random.normal(2, 0.5, n_rollouts)[:, np.newaxis]    #generate new return-go-estimates via rollouts -- IRL this would involve running the env
    if n_upsample > 0:    #if you want to upsample the new data to reduce the contribution of the old data...
        data_new_upsample = np.random.choice(a = data_new.flatten(),size = n_upsample)[:, np.newaxis]    #upsample
        data_new = np.concatenate([data_new,data_new_upsample])    #add upsampled data
    entropy_min = np.min(data_old)    #pull out old data min
    entropy_max = np.max(data_old)    #pull out old data max
    entropy_range = entropy_max - entropy_min    #measure the range of the data
    n_entropy = int(np.ceil((n_samples + n_rollouts)*entropy))    #compute number of samples to add for entropy management
    print(n_entropy)
    if n_entropy > 1:
        # entropy_grid = np.linspace(start = entropy_min - entropy_range*0.1,stop = entropy_max + entropy_range*0.1,num = n_entropy).reshape(n_entropy,1)    #generate entropy grid
        entropy_grid = np.random.uniform(low = entropy_min - entropy_range*0.1,high = entropy_max + entropy_range*0.1,size = n_entropy)[:, np.newaxis]    #generate entropy grid
        data = np.concatenate([data_old,data_new,entropy_grid])    #make data for new KDE estimate
    else:
        data = np.concatenate([data_old,data_new])    #make data for new KDE estimate
    d_G_s_a = KernelDensity(bandwidth=0.75, kernel='gaussian')    #instantiate new KDE
    d_G_s_a.fit(data)    #fit the KDE
    return d_G_s_a


In [ ]:
#@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@
#let's try this caching process out
#@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@
#set up parameters
gamma = 0.9    #set the discount factor
min_reward = -2    #set the minimum period reward
max_reward = 5    #set the max period reward
lower_G_bound = min_reward/(1-gamma)    #compute lower bound on G -- min reward forever
upper_G_bound = max_reward/(1-gamma)    #compute the upper bound on G -- max reward forever

#initialize process
d_G_s_a = d_G_init(low = lower_G_bound,hi = upper_G_bound)    #init the distribution over G assuming uniform

#update distribution from repeated visits
n_rollouts = 5    #how many rollouts
policy_change = [2.0]*10 + [20.0]*10 + [-10]*10 + [30]*10
for mu in policy_change:    #assume multiple visits, with changing policies

    #visualize updated d_G_s_a
    x_grid = np.linspace(lower_G_bound, upper_G_bound, 100)[:, np.newaxis]    #create grid for viz, not need if no viz
    log_pdf = d_G_s_a.score_samples(x_grid)    #create pdf for viz (returns log probability density), not needed if no viz
    plt.plot(x_grid[:, 0], np.exp(log_pdf))    #plot the updated d_G_s_a
    plt.show()    #display to user
    time.sleep(0.75)    #pause so user can see
    clear_output(wait=True)    #clears previous output

    #update d_G_s_a
    data_new = np.random.normal(loc = mu, scale = 0.1, size = n_rollouts)[:, np.newaxis]    #generate new return-go-estimates via rollouts -- IRL this would involve running the env
    d_G_s_a = d_G_update(d_G_s_a,data_new = data_new,n_upsample = 10,n_samples = 50,entropy = 5)    #update using sampling approach


In [ ]:
policy_change

In [ ]:
np.random.uniform(0,1,10)
